Notebooks 18 and 19 covered the Delta Lake transaction model, mutations, and time travel. This notebook focuses on **making Delta tables fast** — how Spark reads less data, compacts small files, and uses indexes to skip entire file groups without scanning them.

```
Query: SELECT * FROM card_txns WHERE category = 'FOOD' AND amount > 500

Without optimization:  scan ALL files → filter in memory

With optimization:
  Partition pruning → skip non-FOOD partition directories
  Data skipping     → skip files where max(amount) < 500
  Z-ORDER           → co-locate FOOD+high-amount rows in fewest files
  OPTIMIZE          → fewer, larger files → less per-file overhead
  Bloom filter      → skip files that cannot contain a specific txn_id
```

In this notebook you will learn:
- **`OPTIMIZE`** — compacting many small files into fewer large ones
- **`ZORDER BY`** — multi-dimensional clustering to maximise data skipping
- **Data skipping** — how Delta uses per-file column statistics to skip files at query time
- **Bloom filter indexes** — probabilistic point-lookup acceleration
- **Partition strategies** — choosing partition columns, over-partitioning pitfalls
- **Generated columns** — computing partition keys automatically from existing columns
- **Table constraints** — `NOT NULL` and `CHECK` constraints
- **Auto-compaction and optimized writes** — Databricks automatic file management
- **`ANALYZE TABLE`** — collecting column-level statistics for the query optimizer
- **`CLONE`** — shallow and deep table copies for testing and branching

In [ ]:
import os, json, shutil, time
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import (
    col, count, sum as _sum, avg, lit, current_timestamp,
    to_date, expr, when, floor, rand, concat, lpad, hash
)
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .appName("DeltaOperationsOptimization")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

BASE    = os.path.dirname(os.path.abspath("20-delta-operations-optimization.ipynb"))
DATA    = os.path.join(BASE, "data")
SCRATCH = os.path.join(BASE, "_scratch_20")
os.makedirs(SCRATCH, exist_ok=True)

print(f"Spark {spark.version} ready")

In [ ]:
# Load source data
card_txns = spark.read.schema(StructType([
    StructField("txn_id",      StringType(),      False),
    StructField("card_id",     StringType(),      False),
    StructField("customer_id", StringType(),      False),
    StructField("amount",      DecimalType(18,2), False),
    StructField("merchant",    StringType(),      True),
    StructField("category",    StringType(),      True),
    StructField("txn_ts",      TimestampType(),   False),
    StructField("status",      StringType(),      False),
    StructField("is_fraud",    BooleanType(),     False),
])).parquet(f"{DATA}/card_transactions")

customers = spark.read.option("header", "true").schema(StructType([
    StructField("customer_id",   StringType(),    False),
    StructField("full_name",     StringType(),    False),
    StructField("city",          StringType(),    True),
    StructField("state",         StringType(),    True),
    StructField("kyc_verified",  BooleanType(),   False),
    StructField("created_at",    TimestampType(), False),
])).csv(f"{DATA}/customers")

print(f"card_txns : {card_txns.count():>6} rows")
print(f"customers : {customers.count():>6} rows")

### The Small-File Problem

Every Spark write job produces files — one file per output partition per task. Streaming pipelines make this worse: each micro-batch adds another set of small files. Over time a table can accumulate thousands of tiny files.

**Why small files hurt performance:**

```
1 000 files × 1 MB each = 1 GB total

Opening a file in S3 costs ~10 ms (LIST + GET metadata + open)
1 000 files × 10 ms = 10 seconds of overhead before reading a single row

Same data in 10 files × 100 MB:
10 files × 10 ms = 0.1 seconds overhead
```

Small files also reduce the effectiveness of Parquet's row-group statistics — fewer rows per file means the min/max range per file is wider, so data skipping skips fewer files.

`OPTIMIZE` solves the small-file problem by rewriting many small files into fewer large ones (target: 1 GB per file).

In [ ]:
# Simulate the small-file problem: write card_txns in 32 tiny partitions
SMALL_FILES_PATH = os.path.join(SCRATCH, "txns_small_files")

(
    card_txns
    .repartition(32)   # forces 32 output files — way too many for this data size
    .write
    .format("delta")
    .mode("overwrite")
    .save(SMALL_FILES_PATH)
)

def count_data_files(path: str) -> tuple:
    """Count parquet files and total size (bytes) in a Delta table directory."""
    n, total = 0, 0
    for root, dirs, files in os.walk(path):
        if "_delta_log" in root:
            continue
        for f in files:
            if f.endswith(".parquet"):
                n += 1
                total += os.path.getsize(os.path.join(root, f))
    return n, total

n_before, sz_before = count_data_files(SMALL_FILES_PATH)
print(f"Before OPTIMIZE: {n_before} files, {sz_before / 1024:.1f} KB total")

### OPTIMIZE — File Compaction

`OPTIMIZE` reads small files in a partition, combines their rows, and writes a smaller number of larger files (target size: 1 GB). The old files are marked `remove` in the log; the new compacted files are added. The operation is fully atomic — readers see either all old files or all new files, never a mix.

```sql
OPTIMIZE table_name                          -- compact all partitions
OPTIMIZE table_name WHERE date = '2024-01-01' -- compact one partition only
OPTIMIZE table_name ZORDER BY (col1, col2)   -- compact + cluster in one step
```

**When to run OPTIMIZE:**
- After a streaming pipeline accumulates many small files
- After many small `append` writes or `MERGE` operations
- On a schedule (e.g., nightly) for tables written to continuously
- Before running a large analytical query that will scan the whole table

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS txns_small
    USING delta
    LOCATION '{SMALL_FILES_PATH}'
""")

t0 = time.time()
optimize_result = spark.sql("OPTIMIZE txns_small")
t_opt = time.time() - t0

optimize_result.select(
    col("metrics.numFilesAdded").alias("files_added"),
    col("metrics.numFilesRemoved").alias("files_removed"),
    col("metrics.filesAdded.avg").alias("avg_new_file_bytes"),
    col("metrics.filesRemoved.avg").alias("avg_old_file_bytes"),
).show()

n_after, sz_after = count_data_files(SMALL_FILES_PATH)
print(f"Before : {n_before} files,  {sz_before / 1024:.1f} KB")
print(f"After  : {n_after}  files,  {sz_after  / 1024:.1f} KB")
print(f"OPTIMIZE completed in {t_opt:.1f}s")

In [ ]:
# Measure read performance before and after OPTIMIZE
# (Re-create the fragmented version for a fair comparison)
FRAG_PATH    = os.path.join(SCRATCH, "txns_fragmented")
COMPACT_PATH = os.path.join(SCRATCH, "txns_compacted")

card_txns.repartition(32).write.format("delta").mode("overwrite").save(FRAG_PATH)
card_txns.repartition(32).write.format("delta").mode("overwrite").save(COMPACT_PATH)

spark.sql(f"OPTIMIZE delta.`{COMPACT_PATH}`")

pred = col("amount") > 500

t0 = time.time()
n_frag = spark.read.format("delta").load(FRAG_PATH).filter(pred).count()
t_frag = time.time() - t0

t0 = time.time()
n_comp = spark.read.format("delta").load(COMPACT_PATH).filter(pred).count()
t_comp = time.time() - t0

print(f"Fragmented ({n_before} files) : {n_frag} rows in {t_frag:.2f}s")
print(f"Compacted  ({n_after}  files) : {n_comp} rows in {t_comp:.2f}s")

### Data Skipping — How Delta Avoids Reading Files

When Delta writes a Parquet file it records column-level statistics in the transaction log for each file:
- `minValues` — minimum value for each column in that file
- `maxValues` — maximum value for each column in that file
- `nullCount` — number of nulls per column
- `numRecords` — total rows in the file

At query time, if a filter predicate is provably false for a file based on its stats, Spark skips that file entirely without opening it.

```
Query: WHERE amount > 500

File A: min=10, max=490   → max < 500 → SKIP (no row can satisfy amount > 500)
File B: min=50, max=650   → max > 500 → READ (may contain matching rows)
File C: min=10, max=300   → max < 500 → SKIP
File D: min=200, max=800  → max > 500 → READ

Only 2 of 4 files read — 50% I/O savings.
```

**Data skipping is most effective when** the data is physically sorted or clustered by the filter column — because then min/max ranges are narrow and more files can be skipped. This is exactly what `ZORDER BY` achieves.

In [ ]:
# Inspect per-file statistics stored in the Delta log
import json as _json

STATS_PATH = os.path.join(SCRATCH, "txns_stats")
card_txns.repartition(4).write.format("delta").mode("overwrite").save(STATS_PATH)

log_dir  = os.path.join(STATS_PATH, "_delta_log")
log_file = os.path.join(log_dir, "00000000000000000000.json")

print("Per-file statistics in the transaction log:\n")
with open(log_file) as f:
    for line in f:
        entry = _json.loads(line)
        if "add" in entry:
            stats = _json.loads(entry["add"].get("stats", "{}"))
            path  = entry["add"]["path"].split("/")[-1][:30]
            print(f"File: {path}")
            print(f"  numRecords : {stats.get('numRecords')}")
            mv = stats.get('minValues', {})
            xv = stats.get('maxValues', {})
            for c in ["amount", "txn_ts"]:
                if c in mv:
                    print(f"  {c}: min={mv[c]}  max={xv[c]}")
            print()

### ZORDER BY — Multi-Dimensional Clustering

Z-ordering is a technique that **co-locates related data in the same files** using a space-filling curve. When you `OPTIMIZE … ZORDER BY (col1, col2)`, Delta sorts rows using a Z-order key derived from the specified columns and packs them into files. The result: files that contain narrow ranges of col1 **and** col2 values simultaneously.

**Why not just sort?**

A regular sort on `(category, amount)` clusters data well for `category` filters but not for `amount` filters alone. Z-ordering interleaves the sort keys so that both `category` and `amount` benefit from data skipping simultaneously.

```
Regular sort by category, amount:
  File 1: ENTERTAINMENT 10–200    ← skipped for amount > 500
  File 2: ENTERTAINMENT 201–600   ← must read (max > 500)
  File 3: FOOD 10–350             ← skipped for amount > 500
  File 4: FOOD 351–800            ← must read
  ...
  Skippable for category filter ✓  but 50% of files must be read for amount > 500

Z-ORDER BY (category, amount):
  File 1: ENTERTAINMENT+FOOD 10–150   ← skipped for amount > 500
  File 2: ENTERTAINMENT+FOOD 450–800  ← read
  ...
  Better skipping on BOTH dimensions simultaneously
```

**Rules for choosing ZORDER columns:**
- Use columns that appear in `WHERE` clauses frequently
- Use columns with **high cardinality** (customer_id, txn_id) — Z-ordering on a boolean is useless
- 2–4 columns maximum — more columns dilutes the benefit
- Do **not** Z-ORDER by partition columns — they already direct readers to the right directory
- Z-ORDER is re-run every time `OPTIMIZE` runs — it is not maintained incrementally

In [ ]:
ZORDER_PATH = os.path.join(SCRATCH, "txns_zorder")
NOORDER_PATH = os.path.join(SCRATCH, "txns_noorder")

# Both tables start with the same fragmented data
for path in [ZORDER_PATH, NOORDER_PATH]:
    card_txns.repartition(16).write.format("delta").mode("overwrite").save(path)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS txns_zorder
    USING delta LOCATION '{ZORDER_PATH}'
""")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS txns_noorder
    USING delta LOCATION '{NOORDER_PATH}'
""")

# OPTIMIZE without Z-ORDER on the baseline
spark.sql("OPTIMIZE txns_noorder")

# OPTIMIZE + ZORDER BY on the experimental table
zorder_result = spark.sql("""
    OPTIMIZE txns_zorder
    ZORDER BY (customer_id, amount)
""")
zorder_result.select(
    col("metrics.numFilesAdded").alias("files_added"),
    col("metrics.numFilesRemoved").alias("files_removed"),
    col("metrics.numBytesAdded").alias("bytes_added"),
).show()
print("ZORDER BY (customer_id, amount) applied.")

In [ ]:
# Compare query performance: filter on customer_id + amount
# Z-ordered table should skip more files

TARGET_CUSTOMER = "CUST0005"
pred = (col("customer_id") == TARGET_CUSTOMER) & (col("amount") > 400)

# Warm up
spark.read.format("delta").load(NOORDER_PATH).filter(pred).count()
spark.read.format("delta").load(ZORDER_PATH).filter(pred).count()

# Benchmark
t0 = time.time()
n_no  = spark.read.format("delta").load(NOORDER_PATH).filter(pred).count()
t_no  = time.time() - t0

t0 = time.time()
n_zo  = spark.read.format("delta").load(ZORDER_PATH).filter(pred).count()
t_zo  = time.time() - t0

print(f"Filter: customer_id = '{TARGET_CUSTOMER}' AND amount > 400")
print(f"  OPTIMIZE only    : {n_no} rows in {t_no:.2f}s")
print(f"  OPTIMIZE+ZORDER  : {n_zo} rows in {t_zo:.2f}s")
print()
print("Check Spark UI → SQL → Physical Plan → look for 'skipped' files in the FileScan operator")

In [ ]:
# ZORDER by partition column is redundant — partition pruning already handles this
PART_PATH = os.path.join(SCRATCH, "txns_partitioned")

(
    card_txns
    .withColumn("txn_date", to_date(col("txn_ts")))
    .write
    .format("delta")
    .partitionBy("txn_date")   # partition by date — already provides date-level data skipping
    .mode("overwrite")
    .save(PART_PATH)
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS txns_partitioned
    USING delta LOCATION '{PART_PATH}'
""")

# ZORDER within each partition by customer_id + amount
# Do NOT include txn_date in ZORDER — it's already the partition column
spark.sql("""
    OPTIMIZE txns_partitioned
    ZORDER BY (customer_id, amount)
""")

# Partition pruning + Z-ORDER skipping together
spark.sql("""
    EXPLAIN EXTENDED
    SELECT count(*)
    FROM   txns_partitioned
    WHERE  txn_date = '2024-01-15'
    AND    customer_id = 'CUST0005'
    AND    amount > 400
""").select("plan").show(1, truncate=False)

### Bloom Filter Indexes

A **Bloom filter** is a probabilistic data structure that answers the question: "Is this value in this file?" with two possible answers:
- **Definitely not** — skip the file entirely
- **Possibly yes** — read the file (false positives are possible)

Bloom filters are stored per-file in the `_delta_log/` directory and are extremely compact — a few KB per file even for millions of distinct values.

**When to use Bloom filters:**
- High-cardinality columns (UUID, transaction ID, hash)
- Equality filters (`txn_id = 'TX001234'`) — Z-ORDER's min/max stats are less effective here because a UUID has a wide min/max range but only one file contains any given value
- Do NOT use for low-cardinality or range filters — Bloom filters only help equality checks

```sql
CREATE BLOOMFILTER INDEX ON TABLE txns
FOR COLUMNS (txn_id OPTIONS (fpp=0.01, numItems=1000000))
```

- `fpp` — false positive probability (0.01 = 1% chance of reading a file that doesn't contain the value)
- `numItems` — expected number of distinct values in the column (used to size the filter)

In [ ]:
BLOOM_PATH = os.path.join(SCRATCH, "txns_bloom")

card_txns.write.format("delta").mode("overwrite").save(BLOOM_PATH)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS txns_bloom
    USING delta LOCATION '{BLOOM_PATH}'
""")

# Create a Bloom filter index on txn_id (UUID-style high-cardinality column)
spark.sql("""
    CREATE BLOOMFILTER INDEX ON TABLE txns_bloom
    FOR COLUMNS (
        txn_id  OPTIONS (fpp=0.01, numItems=50000),
        card_id OPTIONS (fpp=0.05, numItems=10000)
    )
""")

print("Bloom filter index created.")

# The index is applied during OPTIMIZE — compact first to get the filters
spark.sql("OPTIMIZE txns_bloom ZORDER BY (txn_id)")

print("_delta_log contents (look for bloom filter files):")
log_dir = os.path.join(BLOOM_PATH, "_delta_log")
for f in sorted(os.listdir(log_dir))[:6]:
    size = os.path.getsize(os.path.join(log_dir, f))
    print(f"  {f}  ({size:,} bytes)")

In [ ]:
# Point-lookup benchmark: find a specific transaction by ID
sample_id = card_txns.select("txn_id").limit(1).collect()[0][0]
print(f"Looking up txn_id = '{sample_id}'")

# Without Bloom filter
t0 = time.time()
n1 = spark.read.format("delta").load(COMPACT_PATH).filter(col("txn_id") == sample_id).count()
t1 = time.time() - t0

# With Bloom filter
t0 = time.time()
n2 = spark.sql(f"SELECT count(*) FROM txns_bloom WHERE txn_id = '{sample_id}'").collect()[0][0]
t2 = time.time() - t0

print(f"  Without Bloom filter : {n1} rows in {t1:.3f}s")
print(f"  With    Bloom filter : {n2} rows in {t2:.3f}s")
print("  (Bloom filter eliminates files that cannot contain this txn_id)")

### Partition Strategies

Partitioning physically separates data into subdirectories, enabling Spark to skip entire directories for filtered queries. Choosing the right partition column is critical — a bad partition choice can hurt performance more than not partitioning at all.

**Rules for choosing partition columns:**

| Rule | Explanation |
|---|---|
| Low-to-medium cardinality | 10–10 000 distinct values. Too few partitions → all data in one directory. Too many → millions of tiny directories. |
| Frequently filtered | Queries must use the partition column in `WHERE` for pruning to work. |
| Balanced distribution | Each partition should have roughly equal row counts. Skewed partitions cause slow tasks. |
| Target 1 GB+ per partition | Less than 1 GB per partition = over-partitioned (small-file problem). |
| Do NOT partition by high-cardinality columns | `customer_id` (millions of values) or `txn_id` (unique) = millions of tiny directories. |

**Most common good partition columns:** date, month, region, status, category.

**Over-partitioning anti-pattern:**
```
card_txns partitioned by customer_id (10 000 unique values):
  10 000 directories × ~50 rows each = terrible for full-table scans
  Listing 10 000 directories is slow even for a simple count(*)
```

In [ ]:
# Good partition: by txn_date (date-level, low cardinality for a year of data)
GOOD_PART_PATH = os.path.join(SCRATCH, "txns_good_partition")

(
    card_txns
    .withColumn("txn_date", to_date(col("txn_ts")))
    .write
    .format("delta")
    .partitionBy("txn_date")
    .mode("overwrite")
    .save(GOOD_PART_PATH)
)

# Check partition count and sizes
partition_dirs = [
    d for d in os.listdir(GOOD_PART_PATH)
    if d.startswith("txn_date=") and os.path.isdir(os.path.join(GOOD_PART_PATH, d))
]
print(f"Partition count: {len(partition_dirs)}")

# Partition pruning in action
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS txns_good_part
    USING delta LOCATION '{GOOD_PART_PATH}'
""")

t0 = time.time()
n = spark.sql("""
    SELECT count(*), sum(amount)
    FROM   txns_good_part
    WHERE  txn_date BETWEEN '2024-01-01' AND '2024-01-07'
""").collect()
print(f"7-day query: {n[0][0]} rows in {time.time()-t0:.2f}s (only 7 partitions read)")

### Generated Columns — Auto-Derived Partition Keys

A **generated column** is a column whose value is automatically computed from other columns by a deterministic expression. Delta evaluates the expression on write — you never need to compute the column yourself.

The most common use: extract a `date` or `year_month` from a timestamp column and use it as the partition key. This way the source data only needs to include the raw timestamp — the partition key is derived automatically.

```sql
CREATE TABLE card_txns (
  txn_id      STRING NOT NULL,
  txn_ts      TIMESTAMP NOT NULL,
  txn_date    DATE GENERATED ALWAYS AS (CAST(txn_ts AS DATE)),
  amount      DECIMAL(18,2),
  ...
)
USING delta
PARTITIONED BY (txn_date)
```

When you write a DataFrame with `txn_ts` but without `txn_date`, Delta computes `txn_date` automatically.

In [ ]:
GEN_COL_PATH = os.path.join(SCRATCH, "txns_gen_col")

# Create a Delta table with a generated column
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS txns_gen_col (
        txn_id      STRING NOT NULL,
        card_id     STRING NOT NULL,
        customer_id STRING NOT NULL,
        amount      DECIMAL(18,2) NOT NULL,
        merchant    STRING,
        category    STRING,
        txn_ts      TIMESTAMP NOT NULL,
        status      STRING,
        is_fraud    BOOLEAN,
        txn_date    DATE GENERATED ALWAYS AS (CAST(txn_ts AS DATE))
    )
    USING delta
    PARTITIONED BY (txn_date)
    LOCATION '{GEN_COL_PATH}'
""")

# Write WITHOUT the txn_date column — Delta computes it automatically
card_txns.write.format("delta").mode("append").saveAsTable("txns_gen_col")

# Confirm txn_date was generated
spark.sql("""
    SELECT txn_id, txn_ts, txn_date
    FROM   txns_gen_col
    LIMIT  5
""").show()

# Partition pruning works automatically with the generated column
spark.sql("""
    SELECT count(*) AS n, round(sum(amount), 2) AS total
    FROM   txns_gen_col
    WHERE  txn_date = '2024-03-15'
""").show()

### Table Constraints

Delta Lake supports two types of constraints that are enforced at write time — any write that violates a constraint is rejected with an error:

| Constraint type | SQL | Description |
|---|---|---|
| `NOT NULL` | Declared in `CREATE TABLE` column definition | Column cannot contain null values |
| `CHECK` | `ALTER TABLE ADD CONSTRAINT name CHECK (expr)` | Arbitrary boolean expression must be true for every row |

Constraints are stored as table properties in the transaction log and enforced by every writer — not just one job. They act as a schema-level data quality gate.

Unlike schema enforcement (which checks data types and column presence), constraints check **value-level** rules: amounts must be positive, status must be a valid enum value, end date must be after start date.

In [ ]:
CONST_PATH = os.path.join(SCRATCH, "loans_constraints")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS loans_constrained (
        loan_id       STRING NOT NULL,
        customer_id   STRING NOT NULL,
        loan_type     STRING NOT NULL,
        principal     DECIMAL(18,2) NOT NULL,
        interest_rate DOUBLE NOT NULL,
        tenure_months INT NOT NULL,
        disbursed_on  DATE NOT NULL,
        status        STRING NOT NULL
    )
    USING delta
    LOCATION '{CONST_PATH}'
""")

# Add CHECK constraints
spark.sql("""
    ALTER TABLE loans_constrained
    ADD CONSTRAINT principal_positive CHECK (principal > 0)
""")
spark.sql("""
    ALTER TABLE loans_constrained
    ADD CONSTRAINT interest_range CHECK (interest_rate BETWEEN 1.0 AND 30.0)
""")
spark.sql("""
    ALTER TABLE loans_constrained
    ADD CONSTRAINT valid_status CHECK (status IN ('ACTIVE','CLOSED','DELINQUENT','UNDER_REVIEW'))
""")

print("Constraints added:")
spark.sql("SHOW TBLPROPERTIES loans_constrained") \
    .filter(col("key").startswith("delta.constraints")).show(truncate=False)

In [ ]:
from pyspark.sql.utils import AnalysisException

# Good write — all constraints satisfied
good_loans = spark.read.option("multiLine", "true").schema(StructType([
    StructField("loan_id",       StringType(),      False),
    StructField("customer_id",   StringType(),      False),
    StructField("loan_type",     StringType(),      False),
    StructField("principal",     DecimalType(18,2), False),
    StructField("interest_rate", DoubleType(),      False),
    StructField("tenure_months", IntegerType(),     False),
    StructField("disbursed_on",  DateType(),        False),
    StructField("status",        StringType(),      False),
])).json(f"{DATA}/loan_accounts")

good_loans.write.format("delta").mode("append").saveAsTable("loans_constrained")
print(f"Good write succeeded: {spark.sql('SELECT count(*) FROM loans_constrained').collect()[0][0]} rows")

# Bad write 1 — negative principal
bad1 = spark.createDataFrame([
    ("LN-BAD-001", "CUST0001", "PERSONAL", -5000.0, 10.0, 24, "2024-01-01", "ACTIVE")
], ["loan_id","customer_id","loan_type","principal","interest_rate","tenure_months","disbursed_on","status"])
try:
    bad1.write.format("delta").mode("append").saveAsTable("loans_constrained")
except Exception as e:
    print(f"\nBad write 1 REJECTED (negative principal): {str(e)[:120]}")

# Bad write 2 — invalid status
bad2 = spark.createDataFrame([
    ("LN-BAD-002", "CUST0002", "HOME", 500000.0, 9.5, 120, "2024-01-01", "PENDING")
], ["loan_id","customer_id","loan_type","principal","interest_rate","tenure_months","disbursed_on","status"])
try:
    bad2.write.format("delta").mode("append").saveAsTable("loans_constrained")
except Exception as e:
    print(f"Bad write 2 REJECTED (invalid status): {str(e)[:120]}")

### ANALYZE TABLE — Collecting Statistics

`ANALYZE TABLE` computes column-level statistics (distinct count, min, max, null count, histogram) and stores them in the metastore. These statistics are used by the Spark query optimizer (Catalyst) to choose the best join strategy and estimate result sizes.

Without statistics, Catalyst relies on rough estimates and may choose a sort-merge join when a broadcast join would be faster. After `ANALYZE TABLE`, it has accurate size information to make better decisions.

```sql
ANALYZE TABLE loans COMPUTE STATISTICS              -- row count + table size
ANALYZE TABLE loans COMPUTE STATISTICS FOR ALL COLUMNS  -- full column stats
ANALYZE TABLE loans COMPUTE STATISTICS FOR COLUMNS status, loan_type  -- specific columns
```

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS loans_analyze
    USING delta
    AS SELECT * FROM loans_constrained
""")

# Basic statistics — row count and table size
spark.sql("ANALYZE TABLE loans_analyze COMPUTE STATISTICS")

# Column-level statistics — used for join strategy selection
spark.sql("""
    ANALYZE TABLE loans_analyze
    COMPUTE STATISTICS FOR COLUMNS
    status, loan_type, principal, interest_rate
""")

# View the collected statistics
spark.sql("DESCRIBE EXTENDED loans_analyze") \
    .filter(col("col_name").isin(
        "Statistics", "status", "loan_type", "principal", "interest_rate"
    )).show(truncate=False)

# Column stats detail
spark.sql("DESCRIBE EXTENDED loans_analyze status").show(truncate=False)

### Auto-Compaction and Optimized Writes

On Databricks, two Delta features manage file sizes automatically without needing explicit `OPTIMIZE` calls:

**Auto-compaction** — after each write, if enough small files have accumulated in a partition, Delta automatically runs a compaction in the background. The table is always in a reasonably compacted state.

```python
# Enable at the session level
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

# Or per table
ALTER TABLE loans SET TBLPROPERTIES ('delta.autoOptimize.autoCompact' = 'true')
```

**Optimized writes** — when writing with shuffle (partitioning), Delta coalesces output files to target 1 GB each instead of creating one file per Spark task. This prevents the small-file problem at write time.

```python
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")

# Or per table
ALTER TABLE loans SET TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
```

Both features are Databricks-specific. On open-source Spark, run `OPTIMIZE` on a schedule instead.

In [ ]:
# Enable optimized writes for this session (has effect on Databricks clusters)
try:
    spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
    spark.conf.set("spark.databricks.delta.autoCompact.enabled",   "true")
    print("Optimized writes and auto-compaction enabled (Databricks).")
except Exception:
    print("Note: optimizeWrite and autoCompact are Databricks-only features.")
    print("On open-source Spark, run OPTIMIZE on a schedule instead.")

# Show the equivalent table property approach
print()
print("Equivalent table-level properties (set once, persisted):")
print("  delta.autoOptimize.optimizeWrite = true")
print("  delta.autoOptimize.autoCompact   = true")

### CLONE — Shallow and Deep Copies

`CLONE` creates a copy of a Delta table. Two modes:

| Mode | Cost | Data files copied | Use for |
|---|---|---|---|
| **Shallow clone** | Very cheap (metadata only) | No — references source files | Dev/test, short-lived branches, exploring data without affecting prod |
| **Deep clone** | Expensive (copies all files) | Yes — fully independent copy | DR, archiving, migrating tables to a different location |

**Shallow clone** is particularly useful: it creates a full working Delta table (with transaction log) that references the source files. You can run `DELETE`, `UPDATE`, `MERGE` on the clone without affecting the source — Delta copies files on write (copy-on-write semantics).

```sql
CREATE TABLE loans_clone  SHALLOW CLONE loans AT VERSION AS OF 3
CREATE TABLE loans_backup DEEP    CLONE loans
```

In [ ]:
SHALLOW_PATH = os.path.join(SCRATCH, "loans_shallow_clone")
DEEP_PATH    = os.path.join(SCRATCH, "loans_deep_clone")

# Shallow clone — instantaneous, no data copied
t0 = time.time()
spark.sql(f"""
    CREATE OR REPLACE TABLE loans_shallow
    SHALLOW CLONE loans_constrained
    LOCATION '{SHALLOW_PATH}'
""")
t_shallow = time.time() - t0

# Deep clone — copies all data files
t0 = time.time()
spark.sql(f"""
    CREATE OR REPLACE TABLE loans_deep
    DEEP CLONE loans_constrained
    LOCATION '{DEEP_PATH}'
""")
t_deep = time.time() - t0

print(f"Shallow clone: {t_shallow:.2f}s  — metadata only")
print(f"Deep    clone: {t_deep:.2f}s  — all files copied")

print(f"\nShallow clone rows: {spark.sql('SELECT count(*) FROM loans_shallow').collect()[0][0]}")
print(f"Deep    clone rows: {spark.sql('SELECT count(*) FROM loans_deep').collect()[0][0]}")

In [ ]:
# Mutations on a shallow clone do NOT affect the source table
print("Source rows before:", spark.sql("SELECT count(*) FROM loans_constrained").collect()[0][0])

# Delete from the shallow clone
spark.sql("DELETE FROM loans_shallow WHERE status = 'CLOSED'")

shallow_count = spark.sql("SELECT count(*) FROM loans_shallow").collect()[0][0]
source_count  = spark.sql("SELECT count(*) FROM loans_constrained").collect()[0][0]

print(f"Shallow clone after DELETE : {shallow_count}  (closed rows removed)")
print(f"Source table unchanged     : {source_count}  (not affected)")

### Optimization Decision Guide

Use this guide to pick the right optimization for your workload:

```
Problem: queries are slow on a large Delta table
│
├── Many small files? (> 100 files, each < 128 MB)
│   └── OPTIMIZE table  ← compacts to ~1 GB files
│
├── Filters on specific columns (category, date range)?
│   ├── Low-cardinality filter column → PARTITIONED BY
│   └── Medium-cardinality (customer, merchant) → OPTIMIZE ZORDER BY
│
├── Point lookups by UUID / txn_id?
│   └── BLOOMFILTER INDEX on the column
│
├── Join strategy is wrong (sort-merge when broadcast would work)?
│   └── ANALYZE TABLE … COMPUTE STATISTICS FOR COLUMNS
│
├── Need a dev/test copy without paying for data duplication?
│   └── SHALLOW CLONE
│
├── Writing from streaming → constant small files?
│   ├── Databricks: enable autoCompact + optimizeWrite table properties
│   └── Open-source: schedule OPTIMIZE after each stream checkpoint
│
└── Data quality issues reaching the table?
    └── ADD CONSTRAINT (CHECK / NOT NULL)
```

**Combining strategies** — the most effective setup for an analytical table:
1. `PARTITIONED BY (date)` — directory-level skipping for time-range queries
2. `OPTIMIZE ZORDER BY (customer_id, category)` — within-partition clustering for common filters
3. `BLOOMFILTER INDEX` on `txn_id` — fast point lookups
4. `VACUUM RETAIN 168 HOURS` on a weekly schedule — reclaim disk space

In [ ]:
# Full optimization pipeline on a single table
FULL_OPT_PATH = os.path.join(SCRATCH, "txns_full_opt")

# Step 1: Write with sensible partitioning
(
    card_txns
    .withColumn("txn_date", to_date(col("txn_ts")))
    .repartition(16)   # simulate fragmented writes
    .write
    .format("delta")
    .partitionBy("txn_date")
    .mode("overwrite")
    .save(FULL_OPT_PATH)
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS txns_full_opt
    USING delta LOCATION '{FULL_OPT_PATH}'
""")

n_pre, sz_pre = count_data_files(FULL_OPT_PATH)
print(f"Before optimization: {n_pre} files, {sz_pre/1024:.0f} KB")

# Step 2: OPTIMIZE + ZORDER within each partition
spark.sql("""
    OPTIMIZE txns_full_opt
    ZORDER BY (customer_id, amount)
""")

# Step 3: Bloom filter on txn_id for point lookups
spark.sql("""
    CREATE BLOOMFILTER INDEX ON TABLE txns_full_opt
    FOR COLUMNS (txn_id OPTIONS (fpp=0.01, numItems=100000))
""")

# Step 4: Collect statistics for the optimizer
spark.sql("""
    ANALYZE TABLE txns_full_opt
    COMPUTE STATISTICS FOR COLUMNS customer_id, category, amount, status
""")

n_post, sz_post = count_data_files(FULL_OPT_PATH)
print(f"After  optimization: {n_post} files, {sz_post/1024:.0f} KB")

# Step 5: VACUUM to reclaim old file space (7-day retention for prod)
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
spark.sql("VACUUM txns_full_opt RETAIN 0 HOURS")
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")

n_vac, sz_vac = count_data_files(FULL_OPT_PATH)
print(f"After  VACUUM:       {n_vac} files, {sz_vac/1024:.0f} KB  (old files removed)")

### Summary

**`OPTIMIZE`** compacts many small files into fewer large ones (target ~1 GB each). Run after streaming writes, many small appends, or before large analytical queries. The operation is atomic — no reader sees a partial state.

**`ZORDER BY`** co-locates rows with similar values in the same files using a space-filling curve. Maximises data skipping for multi-column filters. Use for medium-to-high cardinality columns that appear in `WHERE` clauses. Do not Z-ORDER by partition columns.

**Data skipping** uses per-file min/max/null statistics stored in the transaction log. Spark skips files whose stats prove no row can satisfy the filter. More effective after OPTIMIZE (fewer, larger files with tighter stat ranges).

**Bloom filter indexes** accelerate equality point lookups on high-cardinality columns (UUIDs, txn IDs). Probabilistic — eliminates files that definitely don't contain the value. Does not help range queries.

**Partition strategies** — partition by low-cardinality, frequently-filtered columns targeting ≥1 GB per partition. Over-partitioning (millions of tiny directories) hurts list and scan performance.

**Generated columns** compute partition keys automatically from raw columns (e.g., `txn_date` from `txn_ts`). Writers only provide the raw column; Delta fills in the derived value.

**Table constraints** (`NOT NULL`, `CHECK`) enforce value-level data quality at write time — any write violating a constraint is rejected atomically.

**`ANALYZE TABLE`** collects column statistics for Catalyst, enabling accurate join strategy selection and cardinality estimates.

**`CLONE`** — shallow clone is free (metadata only, references source files, copy-on-write on mutation); deep clone copies all files for full independence.

**Combined best practice** for an analytical Delta table: `PARTITIONED BY` date + `OPTIMIZE ZORDER BY` common filter columns + `BLOOMFILTER INDEX` on ID columns + `ANALYZE TABLE` for column stats + weekly `VACUUM`.

In [ ]:
# Cleanup
for tbl in [
    "txns_small", "txns_zorder", "txns_noorder", "txns_partitioned",
    "txns_gen_col", "txns_good_part", "loans_constrained",
    "loans_analyze", "loans_shallow", "loans_deep", "txns_bloom",
    "txns_full_opt",
]:
    spark.sql(f"DROP TABLE IF EXISTS {tbl}")

if os.path.exists(SCRATCH):
    shutil.rmtree(SCRATCH)
    print(f"Removed: {SCRATCH}")
print("Cleanup complete.")